# 01. 공인 IP → 사설 IP 전환

| 항목 | 내용 |
|---|---|
| 목표 | 서버의 랜카드를 사설 IP 로 바꾸고 인터넷·DNS 가 되는지 확인 |
| 소요 시간 | 서버당 30분 |
| 시작 전 조건 | 학교에서 받은 값(IP·마스크·게이트웨이·DNS), 서버실 콘솔 |
| 실행 위치 | 🖥 서버 콘솔 (절마다 다시 표시) |

> 대문자 대괄호 `[LIKE_THIS]` 는 **자기 환경의 값으로 바꿔 쓰는 자리표시자**입니다. 표는 [00_overview](00_overview.ipynb)에 있습니다.
> 이 노트북의 셀은 **문서**입니다. 설명대로 해당 위치(서버 콘솔 등)에서 직접 실행하세요. 명령에 따라 서버 설정이 바뀝니다.

## A. 학교에서 받을 값

| 값 | 비고 |
|---|---|
| 서버의 사설 IP | 서버마다 다름 |
| 서브넷 마스크 → `[PREFIX]` | 보통 `/24` |
| 게이트웨이 `[GATEWAY]` | 학교 안내문: **"할당받은 IP 의 세 번째 자리까지 쓰고 마지막은 1"** |
| DNS `[SCHOOL_DNS1]`, `[SCHOOL_DNS2]` | 안내문에서 가려져 있으면 안내문을 받은 PC 에서 `ipconfig /all` 로 확인 |
| 사설 대역에서 인터넷(NAT) 가능 여부 | **안 되면 이후 설치(K3s·GitHub·Tailscale)가 모두 막힘** |

> 안내문에 나온 `10.x.x.254` 같은 번호는 **PC 설정 화면의 예시**일 수 있습니다. 서버에 실제로 부여된 번호인지 반드시 확인하세요.
> 관리 서버부터, 서버를 쓰는 사람이 없는 시간에 하세요. 외부(공인 IP)로 들어오던 SSH 접속이 **끊깁니다** — 미리 공지하세요.

## B. 현재 상태 기록 · 백업 (🖥 서버 콘솔)

In [ ]:
!ip -br link                         # 인터페이스 이름([IFACE]) 확인
!ip -4 addr; ip route                # 현재 주소·게이트웨이 기록
!ls /etc/netplan/
!sudo cp -a /etc/netplan /root/netplan.bak        # 백업
!sudo ufw status numbered            # 현재 방화벽 규칙 기록

## C. 사설 IP 가 이 랜선에 닿는지 **바꾸기 전에** 확인 (재부팅하면 사라지는 임시 주소)

`ping` 에 응답이 오면 스위치 포트가 이미 사설 대역에 열려 있는 것입니다. 응답이 없으면 **전환하지 말고** 학교에 포트 설정을 요청하세요.

In [ ]:
!sudo ip addr add [MGMT_PRIVATE_IP]/[PREFIX] dev [IFACE]
!ping -c2 -W1 -I [MGMT_PRIVATE_IP] [GATEWAY]
!sudo ip addr del [MGMT_PRIVATE_IP]/[PREFIX] dev [IFACE]

## D. netplan 교체

기존 설정(공인 IP·기본 경로)이 새 설정과 겹치면 충돌하므로 **치워 둡니다**(백업은 B 에서 했음).
DNS 는 **학교 DNS 를 먼저**, 공개 DNS 를 뒤에 둡니다. 통신사 공개 DNS(KT 등)는 사설망에서 **막혀 있을 수 있어** 쓰지 않습니다.

In [ ]:
!sudo mkdir -p /root/netplan-disabled
!sudo mv /etc/netplan/*.yaml /root/netplan-disabled/

In [ ]:
%%bash
sudo tee /etc/netplan/50-static.yaml >/dev/null <<'EOF'
network:
  version: 2
  ethernets:
    [IFACE]:
      dhcp4: false
      addresses: [[MGMT_PRIVATE_IP]/[PREFIX]]
      routes:
        - to: default
          via: [GATEWAY]
      nameservers:
        addresses: [[SCHOOL_DNS1], [SCHOOL_DNS2], 1.1.1.1, 8.8.8.8]
EOF
sudo chmod 600 /etc/netplan/50-static.yaml

## E. 적용 — `netplan try` (120초 안에 확정하지 않으면 **자동 원복**)

1. `netplan try` 를 실행합니다. 화면이 `Press ENTER to accept` 에서 멈춥니다.
2. **다른 콘솔**(`Alt+F2` 로 전환해 로그인)에서 아래 F 점검을 합니다.
3. **모두 통과하면** 원래 화면(`Alt+F1`)으로 돌아와 Enter 로 확정합니다. 하나라도 실패하면 **아무것도 누르지 말고** 기다리세요(120초 뒤 자동 원복).

In [ ]:
!sudo netplan try --timeout 120

## F. 인터넷·이름 해석 점검 (확정 전에!)

> `curl` 의 `-o` 는 영문 **오**입니다. `-i` 로 잘못 치면 `000` 이 나옵니다.

In [ ]:
!ping -c3 [GATEWAY]
!ping -c3 1.1.1.1
!getent hosts pypi.org
!curl -s -o /dev/null -w '%{http_code}\n' https://github.com
!curl -s -o /dev/null -w '%{http_code}\n' https://ghcr.io
!curl -s -o /dev/null -w '%{http_code}\n' https://tailscale.com

| 결과 | 의미 |
|---|---|
| ping 모두 성공, `getent` 가 주소 출력, `curl` 이 `200`/`301`/`302`/`405` 류 숫자 | 정상 |
| ping 은 되는데 `getent`/`curl` 실패 | **DNS 문제** — 학교 DNS 번호 확인(통신사 DNS 는 사설망에서 막힘) |
| 모두 실패 | 사설 대역의 인터넷(NAT)이 막힘 → **확정하지 말고** 학교에 확인 |

`ghcr.io` 는 `HEAD` 요청에 `405` 를 돌려주는 사이트입니다. 숫자가 나왔다면 **연결은 된 것**입니다.

## G. 확정 후 확인

In [ ]:
!ip -4 addr show [IFACE]             # 사설 IP 만 남았는지(공인 IP 가 사라졌는지)
!resolvectl status [IFACE] | grep -i "DNS Servers"
!ip route

## H. 방화벽(ufw) — **허용 규칙 먼저, 기본 거부는 맨 마지막**

- SSH 포트(`[SSH_PORT]`)는 학교 사설 대역에서만 허용합니다.
- **Docker 가 연 포트(`-p 8080:80`)는 ufw 를 우회합니다.** `docker ps` 에서 `0.0.0.0:…` 로 열린 포트가 있으면 사설 IP 로만 다시 띄우세요. (이 단계의 관리 서버에는 Docker 가 없을 수 있습니다.)

In [ ]:
!sudo ufw allow from [PRIVATE_SUBNET]/[PREFIX] to any port [SSH_PORT] proto tcp
!sudo ufw status numbered
# 허용 규칙이 모두 보이는지 확인한 뒤 맨 마지막에:
# sudo ufw default deny incoming

## 되돌리기
- 확정 **전**: 아무것도 누르지 않으면 120초 뒤 자동 원복.
- 확정 **후** 문제가 생기면 (콘솔에서):

In [ ]:
!sudo rm /etc/netplan/50-static.yaml
!sudo mv /root/netplan-disabled/*.yaml /etc/netplan/
!sudo netplan apply

## ✅ 완료 확인

- [ ] 서버에 사설 IP 만 있다
- [ ] 게이트웨이 ping 성공
- [ ] `getent hosts pypi.org` 가 주소를 출력한다
- [ ] github·ghcr·tailscale 에 숫자 응답이 나온다
- [ ] ufw 허용 규칙이 있고 기본 거부는 확인 후에 켰다